# 015 — Surgical intervention alpha sweep (G-ALPHA)

This is the new v3 core. The alpha grid, rank<=10 carrying-position rule,
thresholds, and source-capped primary operator were frozen in notebook 00.
A G-SWAP-only pilot then showed the source-capped operator could not flip the
spider case through alpha=2. The existing fractional coordinate swap restricted
to the same clean carrying mask is therefore included as an exploratory,
nonselectable sensitivity analysis; it was not frozen in notebook 00. The
all-position fractional swap is also a nonselectable reference. Random and
absent-coordinate controls are evaluated for every policy/alpha row.

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path('/home/jovyan/j-space-thoughts')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['HF_HOME'] = str(Path.home() / '.cache/huggingface')
os.environ['HF_HUB_CACHE'] = str(Path.home() / '.cache/huggingface/hub')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(Path.home() / '.cache/huggingface/hub')

metrics = json.loads((ROOT / 'results/metrics.json').read_text())
v3 = metrics['calibration_v3']
repair_v2 = metrics['repair_v2']
assert v3['gate_ledger']['stage0_reverify'] == 'PASS'
assert v3['gate_ledger']['g_swap'] == 'PASS'
assert v3['gate_ledger']['g_alpha'] in {'PENDING', 'FAIL'}
assert v3['gate_ledger']['stage3_science'] in {
    'PROHIBITED', 'SKIPPED_PREREQUISITE'
}
workspace_layers = v3['protocol']['workspace_layers']
print(json.dumps({
    'alpha_grid': v3['protocol']['alpha_grid'],
    'position_rule': v3['protocol']['position_rule'],
    'primary_edit': v3['protocol']['primary_edit'],
    'thresholds': v3['protocol']['thresholds'],
}, indent=2))

from src.jlens_iface import load_published_lens
from src.model_utils import load_model
from src.v2_repair import MODEL_ID

bundle = load_model(MODEL_ID)
lens = load_published_lens(MODEL_ID)

{
  "alpha_grid": [
    0.25,
    0.5,
    0.75,
    1.0,
    1.25,
    1.5,
    1.75,
    2.0
  ],
  "position_rule": "original prompt position selected iff source-label J-Lens rank is <=10 at any workspace layer; empty masks remain no-op",
  "primary_edit": "project_out_transfer",
  "thresholds": {
    "absent_abs_null_over_real": 0.25,
    "capability_abs_grand_and_bank_mean_delta_nll": 0.25,
    "g_pos_low_causal_abs_delta": 0.5,
    "g_pos_max_weight_read_ratio": 0.5,
    "g_pos_min_languages": 3,
    "g_pos_min_passages": 6,
    "known_swap_top1": "3/3",
    "random_empirical_p": 0.05
  }
}


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [2]:
from src.v3_alpha_sweep import run_alpha_sweep

sweep = run_alpha_sweep(
    bundle,
    lens,
    repair_v2=repair_v2,
    workspace_layers=workspace_layers,
)
print('mask manifest SHA256', sweep['mask_manifest']['sha256'])
print('Known-answer carrying masks:')
for name, mask in sweep['mask_manifest']['known'].items():
    print(name, mask['positions'], f"{len(mask['positions'])}/{mask['sequence_length']}")
print()
print('Mask-specific primary weight-READ ratios:')
for key, row in sweep['masked_weight_read'].items():
    print(key, row['primary_ratio'], row['automatic_mask']['positions'])
print()
print('Full alpha table:')
for row in sweep['rows']:
    print({
        'policy': row['policy'],
        'alpha': row['alpha'],
        'swaps': row['known_swaps']['n_pass'],
        'mean_delta_nll': row['capability']['mean_delta_nll'],
        'gpos': row['g_pos']['n_reproduced'],
        'random': row['random_null']['status'],
        'absent': row['absent_null']['status'],
        'valid': row['valid'],
    })
print(json.dumps({
    'G-ALPHA': sweep['g_alpha'],
    'selected_intervention': sweep['selected_intervention'],
    'raw_artifact': sweep['raw_artifact'],
    'raw_artifact_sha256': sweep['raw_artifact_sha256'],
    'raw_artifact_bytes': sweep['raw_artifact_bytes'],
    'figure': sweep['figure'],
}, indent=2))

mask manifest SHA256 1882edbe1f73e85de06436924a41d532e6c2f7fb0d0f5deb8a8e6d2dfe7de049
Known-answer carrying masks:
spider-legs [6, 11, 12, 13] 4/14
animal-legs-buffalo2 [10, 11, 17, 21] 4/23
chem-photosynthesis-Z [5, 7, 8, 9, 10, 11, 15] 7/16

Mask-specific primary weight-READ ratios:
fr1 0.8489580574965856 [8, 9, 13, 15, 32]
fr2 1.0 [8, 19]
de1 1.0 [9, 21]
de2 1.1181865076591622 [8, 10, 38]
es1 1.0 [8]
es2 1.0 [9, 38]
it1 1.247494787661422 [8, 29]
it2 1.1210801168485458 [9, 10, 17, 22]

Full alpha table:
{'policy': 'project_out_transfer', 'alpha': 0.25, 'swaps': 0, 'mean_delta_nll': 0.0, 'gpos': 0, 'random': 'FAIL', 'absent': 'FAIL', 'valid': False}
{'policy': 'project_out_transfer', 'alpha': 0.5, 'swaps': 0, 'mean_delta_nll': 0.0, 'gpos': 0, 'random': 'PASS', 'absent': 'PASS', 'valid': False}
{'policy': 'project_out_transfer', 'alpha': 0.75, 'swaps': 0, 'mean_delta_nll': 0.0, 'gpos': 0, 'random': 'PASS', 'absent': 'FAIL', 'valid': False}
{'policy': 'project_out_transfer', 'alpha': 1.

In [3]:
from src.v3_alpha_sweep import persist_alpha_sweep

metrics = persist_alpha_sweep(sweep)
v3 = metrics['calibration_v3']
assert v3['gate_ledger']['g_alpha'] == sweep['g_alpha']
assert v3['gate_ledger']['stage3_science'] in {
    'PROHIBITED', 'SKIPPED_PREREQUISITE'
}
next_notebook = (
    '04_recalibration_at_alpha'
    if sweep['g_alpha'] == 'PASS'
    else '04_recalibration_skip_then_stage4'
)
print(json.dumps({
    'g_alpha': v3['gate_ledger']['g_alpha'],
    'stage2': v3['gate_ledger']['stage2_recalibration'],
    'stage3': v3['gate_ledger']['stage3_science'],
    'next': next_notebook,
}, indent=2))

{
  "g_alpha": "FAIL",
  "stage2": "SKIPPED_PREREQUISITE",
  "stage3": "SKIPPED_PREREQUISITE",
  "next": "04_recalibration_skip_then_stage4"
}


In [4]:
import gc
import torch

del sweep, metrics, lens, bundle
gc.collect()
torch.cuda.empty_cache()
print('Notebook 015 complete. No science was run.')

Notebook 015 complete. No science was run.
